# Flood Map Exploration

This notebook works in steps toward a map of South Sudan + neighbouring countries with all flood points plotted.

**Steps:**
1. Load & combine all flood parquet files (recurring + unusual)
2. Load South Sudan admin2 boundaries
3. Spatial join: assign each flood point to a county (admin2) — **memory-efficient via pixel deduplication**
4. Build the county-level flood summary table
5. *(Next notebook)* Plot the full map with neighbouring countries


## 0. Imports

In [5]:
import glob
import os

import geopandas as gpd
import pandas as pd

## 1. Config

In [6]:
DATA_DIR = os.path.join(
    os.path.dirname(os.getcwd()),  # project root
    "raw_data"
)

ADMIN2_PATH   = os.path.join(DATA_DIR, "Administrative boundaries", "ssd_admin2.geojson")
FLOOD_DIR_REC = os.path.join(DATA_DIR, "flood_masks", "compact_recurring")
FLOOD_DIR_UNU = os.path.join(DATA_DIR, "flood_masks", "compact_unusual")

OUTPUT_DIR = os.path.join(os.path.dirname(os.getcwd()), "results", "flood_map")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Data dir    :", DATA_DIR)
print("Admin2 path :", ADMIN2_PATH)
print("Recurring   :", FLOOD_DIR_REC)
print("Unusual     :", FLOOD_DIR_UNU)

Data dir    : c:\Users\thijn\OneDrive\TU Eindhoven\Data Science 2026-2027\Capstone Data Challenge JBG060\JBG060_ZHL_2026\raw_data
Admin2 path : c:\Users\thijn\OneDrive\TU Eindhoven\Data Science 2026-2027\Capstone Data Challenge JBG060\JBG060_ZHL_2026\raw_data\Administrative boundaries\ssd_admin2.geojson
Recurring   : c:\Users\thijn\OneDrive\TU Eindhoven\Data Science 2026-2027\Capstone Data Challenge JBG060\JBG060_ZHL_2026\raw_data\flood_masks\compact_recurring
Unusual     : c:\Users\thijn\OneDrive\TU Eindhoven\Data Science 2026-2027\Capstone Data Challenge JBG060\JBG060_ZHL_2026\raw_data\flood_masks\compact_unusual


## 2. Load all flood parquet files

Each file contains columns: `date`, `lat`, `lon`, `tile`, `cloud_frac`.
We load both recurring and unusual floods and label them.

In [7]:
def load_flood_parquets(folder, flood_type):
    """Load all parquet files from a folder and add a flood_type label column."""
    files = sorted(glob.glob(os.path.join(folder, "*.parquet")))
    print(f"  Found {len(files)} files in '{os.path.basename(folder)}'")
    dfs = []
    for f in files:
        df = pd.read_parquet(f)
        df["flood_type"] = flood_type
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


print("Loading recurring floods...")
df_rec = load_flood_parquets(FLOOD_DIR_REC, "recurring")

print("Loading unusual floods...")
df_unu = load_flood_parquets(FLOOD_DIR_UNU, "unusual")

# Combine
df_floods = pd.concat([df_rec, df_unu], ignore_index=True)

# Parse dates
df_floods["date"] = pd.to_datetime(df_floods["date"])
df_floods["year"] = df_floods["date"].dt.year

print(f"\nTotal flood observations: {len(df_floods):,}")
print(f"Date range: {df_floods['date'].min().date()} -> {df_floods['date'].max().date()}")
print(f"Flood types: {df_floods['flood_type'].value_counts().to_dict()}")
df_floods.head()

Loading recurring floods...
  Found 52 files in 'compact_recurring'
Loading unusual floods...
  Found 52 files in 'compact_unusual'

Total flood observations: 95,255,114
Date range: 2000-02-28 -> 2025-12-31
Flood types: {'unusual': 69819820, 'recurring': 25435294}


,date,lat,lon,tile,cloud_frac,flood_type,year
0,2000-03-02,2.155208,22.076042,h20v08,0.0,recurring,2000
1,2000-03-02,2.128125,21.519793,h20v08,0.0,recurring,2000
2,2000-03-02,2.086458,21.344791,h20v08,0.0,recurring,2000
3,2000-03-02,2.076042,22.590626,h20v08,0.0,recurring,2000
4,2000-03-02,2.073958,22.582293,h20v08,0.0,recurring,2000


## 3. Preview the spatial extent of the flood data

In [8]:
print("Latitude  range:", df_floods["lat"].min(), "->", df_floods["lat"].max())
print("Longitude range:", df_floods["lon"].min(), "->", df_floods["lon"].max())
print()
print(df_floods[["lat", "lon", "date", "year"]].describe())

Latitude  range: 0.0010416667 -> 9.998959
Longitude range: 20.001041 -> 39.99896

                lat           lon                        date          year
count  9.525511e+07  9.525511e+07                    95255114  9.525511e+07
mean   6.733869e+00  3.215026e+01  2019-12-15 01:06:00.327090  2.019453e+03
min    1.041667e-03  2.000104e+01         2000-02-28 00:00:00  2.000000e+03
25%    4.534375e+00  2.978021e+01         2016-10-29 00:00:00  2.016000e+03
50%    8.209375e+00  3.113854e+01         2022-09-27 00:00:00  2.022000e+03
75%    9.178125e+00  3.593854e+01         2024-02-03 00:00:00  2.024000e+03
max    9.998959e+00  3.999896e+01         2025-12-31 00:00:00  2.025000e+03
std    3.040260e+00  3.404697e+00                         NaN  6.045807e+00


## 4. Load South Sudan admin2 boundaries

In [9]:
gdf_admin2 = gpd.read_file(ADMIN2_PATH)

# Keep only the most useful columns
gdf_admin2 = gdf_admin2[["adm2_name", "adm1_name", "adm2_pcode", "adm1_pcode", "geometry"]].copy()

print(f"Admin2 counties loaded: {len(gdf_admin2)}")
print(f"CRS: {gdf_admin2.crs}")
gdf_admin2.head()

Admin2 counties loaded: 79
CRS: EPSG:4326


,adm2_name,adm1_name,adm2_pcode,adm1_pcode,geometry
0,Juba,Central Equatoria,SS0101,SS01,"POLYGON ((32.12034 5.13695, 32.1199 5.1385, 32..."
1,Kajo-keji,Central Equatoria,SS0102,SS01,"POLYGON ((31.69216 3.95909, 31.69206 3.95919, ..."
2,Lainya,Central Equatoria,SS0103,SS01,"POLYGON ((31.17471 4.37282, 31.17503 4.37347, ..."
3,Morobo,Central Equatoria,SS0104,SS01,"POLYGON ((31.04738 3.81817, 31.04738 3.82137, ..."
4,Terekeka,Central Equatoria,SS0105,SS01,"POLYGON ((32.10548 5.49597, 32.10685 5.55038, ..."


## 5. Memory-efficient spatial join

**Problem:** Joining ~95 million flood observations directly causes a MemoryError.

**Solution:** MODIS flood pixels sit on a fixed grid, so the same `(lat, lon)` location appears across many dates. 
We deduplicate to **unique pixel locations first**, join only those (~thousands vs millions of points), 
then merge the county label back to the full dataset.

This reduces the join size by orders of magnitude.

In [12]:
import pandas as pd

chunk_size = 1_000_000
active_events = {}
completed_events = []

for start in range(0, len(df_floods), chunk_size):

    chunk = df_floods.iloc[start:start + chunk_size][["lat", "lon", "date"]].copy()
    chunk["date"] = pd.to_datetime(chunk["date"])

    for (lat, lon), group in chunk.groupby(["lat", "lon"], sort=False):

        for date in group["date"].sort_values():

            pixel = (lat, lon)

            if pixel not in active_events:
                active_events[pixel] = {
                    "start_date": date,
                    "last_date": date
                }

            else:
                gap = (date - active_events[pixel]["last_date"]).days

                if gap > 3:
                    completed_events.append({
                        "lat": lat,
                        "lon": lon,
                        "start_date": active_events[pixel]["start_date"],
                        "end_date": active_events[pixel]["last_date"]
                    })

                    active_events[pixel] = {
                        "start_date": date,
                        "last_date": date
                    }

                else:
                    active_events[pixel]["last_date"] = date

    print(f"Processed {min(start + chunk_size, len(df_floods)):,} / {len(df_floods):,}")

# Close remaining events
for (lat, lon), event in active_events.items():
    completed_events.append({
        "lat": lat,
        "lon": lon,
        "start_date": event["start_date"],
        "end_date": event["last_date"]
    })

# Create final event dataframe
flood_events = pd.DataFrame(completed_events)

flood_events["duration_days"] = (
    flood_events["end_date"] - flood_events["start_date"]
).dt.days + 1

print(f"\nFlood events found: {len(flood_events):,}")

Processed 1,000,000 / 95,255,114
Processed 2,000,000 / 95,255,114
Processed 3,000,000 / 95,255,114
Processed 4,000,000 / 95,255,114
Processed 5,000,000 / 95,255,114
Processed 6,000,000 / 95,255,114
Processed 7,000,000 / 95,255,114
Processed 8,000,000 / 95,255,114
Processed 9,000,000 / 95,255,114
Processed 10,000,000 / 95,255,114
Processed 11,000,000 / 95,255,114
Processed 12,000,000 / 95,255,114
Processed 13,000,000 / 95,255,114
Processed 14,000,000 / 95,255,114
Processed 15,000,000 / 95,255,114
Processed 16,000,000 / 95,255,114
Processed 17,000,000 / 95,255,114
Processed 18,000,000 / 95,255,114
Processed 19,000,000 / 95,255,114
Processed 20,000,000 / 95,255,114
Processed 21,000,000 / 95,255,114
Processed 22,000,000 / 95,255,114
Processed 23,000,000 / 95,255,114
Processed 24,000,000 / 95,255,114
Processed 25,000,000 / 95,255,114
Processed 26,000,000 / 95,255,114
Processed 27,000,000 / 95,255,114
Processed 28,000,000 / 95,255,114
Processed 29,000,000 / 95,255,114
Processed 30,000,000 / 

In [27]:
flood_events.head(10)

,lat,lon,start_date,end_date,duration_days
0,0.001042,25.538542,2004-02-08,2004-02-08,1
1,0.001042,25.538542,2004-04-21,2004-04-21,1
2,0.001042,30.021875,2014-12-15,2014-12-15,1
3,0.001042,30.021875,2016-01-03,2016-01-03,1
4,0.001042,30.021875,2020-01-23,2020-01-23,1
5,0.001042,30.021875,2020-04-26,2020-04-26,1
6,0.001042,30.021875,2020-05-28,2020-05-28,1
7,0.001042,30.021875,2020-06-29,2020-07-01,3
8,0.001042,30.021875,2020-08-02,2020-08-02,1
9,0.001042,30.021875,2020-12-15,2020-12-15,1


In [29]:
flood_days_per_pixel = (
    flood_events
    .groupby(["lat", "lon"])["duration_days"]
    .sum()
    .reset_index(name="total_flood_days")
)
flood_days_per_pixel.head(10)

,lat,lon,total_flood_days
0,0.001042,25.538542,3
1,0.001042,30.021875,60
2,0.001042,30.023958,81
3,0.001042,30.026041,12
4,0.001042,30.136457,45
5,0.001042,30.296875,5
6,0.001042,31.253124,1
7,0.001042,31.838541,2
8,0.001042,31.840626,1
9,0.001042,31.844791,2


In [ ]:
# Step B: build a small GeoDataFrame from unique pixels only
gdf_pixels = gpd.GeoDataFrame(
    df_unique_pixels,
    geometry=gpd.points_from_xy(df_unique_pixels["lon"], df_unique_pixels["lat"]),
    crs="EPSG:4326"
)

# Ensure same CRS
if gdf_admin2.crs != gdf_pixels.crs:
    gdf_admin2 = gdf_admin2.to_crs(gdf_pixels.crs)

print(f"Running spatial join on {len(gdf_pixels):,} unique pixels...")

gdf_pixels_joined = gpd.sjoin(
    gdf_pixels,
    gdf_admin2[["adm2_name", "adm1_name", "adm2_pcode", "geometry"]],
    how="left",
    predicate="within"
).drop(columns=["index_right"], errors="ignore")

n_in  = gdf_pixels_joined["adm2_name"].notna().sum()
n_out = gdf_pixels_joined["adm2_name"].isna().sum()
print(f"Pixels inside  South Sudan counties : {n_in:,}")
print(f"Pixels outside South Sudan counties : {n_out:,}  (neighbouring countries / border gaps)")

In [ ]:
# Step C: merge county label back to the full flood observations table
# pixel_county maps (lat, lon) -> county info
pixel_county = gdf_pixels_joined[["lat", "lon", "adm2_name", "adm1_name", "adm2_pcode"]]

df_floods_labelled = df_floods.merge(pixel_county, on=["lat", "lon"], how="left")

n_in  = df_floods_labelled["adm2_name"].notna().sum()
n_out = df_floods_labelled["adm2_name"].isna().sum()
print(f"Flood observations inside  South Sudan : {n_in:,}")
print(f"Flood observations outside South Sudan : {n_out:,}")
df_floods_labelled.head()

## 6. Build the county-level flood summary table

For each county we compute:
- Total flood observations (all time)
- Number of distinct flood days
- Number of distinct years flooded
- Breakdown by flood type (recurring vs unusual)

In [ ]:
# Work only with observations that fall inside a county
df_ssd = df_floods_labelled[df_floods_labelled["adm2_name"].notna()].copy()

# Main aggregation per county
county_table = (
    df_ssd.groupby(["adm2_name", "adm1_name", "adm2_pcode"])
    .agg(
        total_flood_obs  =("date", "count"),
        n_flood_days     =("date", "nunique"),
        n_flood_years    =("year", "nunique"),
        first_flood_date =("date", "min"),
        last_flood_date  =("date", "max"),
    )
    .reset_index()
)

# Recurring vs unusual breakdown
flood_type_pivot = (
    df_ssd.groupby(["adm2_name", "flood_type"])["date"]
    .count()
    .unstack(fill_value=0)
    .rename(columns={"recurring": "obs_recurring", "unusual": "obs_unusual"})
    .reset_index()
)

county_table = county_table.merge(flood_type_pivot, on="adm2_name", how="left")
county_table = county_table.sort_values("total_flood_obs", ascending=False).reset_index(drop=True)

print(f"Counties with at least one flood observation: {len(county_table)}")
county_table

## 7. Save results

In [ ]:
# Save county summary table
out_csv = os.path.join(OUTPUT_DIR, "county_flood_summary.csv")
county_table.to_csv(out_csv, index=False)
print("Saved county table        ->", out_csv)

# Save labelled flood observations (no geometry — lightweight)
out_parquet = os.path.join(OUTPUT_DIR, "flood_points_with_county.parquet")
df_floods_labelled.to_parquet(out_parquet, index=False)
print("Saved labelled flood obs  ->", out_parquet)

# Save the unique pixel -> county lookup (useful for the map step)
out_pixels = os.path.join(OUTPUT_DIR, "unique_pixels_with_county.parquet")
gdf_pixels_joined[["lat", "lon", "adm2_name", "adm1_name", "adm2_pcode"]].to_parquet(out_pixels, index=False)
print("Saved pixel-county lookup ->", out_pixels)

print("\n Done! Ready for the next step: plotting the map.")

## 8. Quick preview — top 15 most-flooded counties

In [ ]:
county_table[
    ["adm2_name", "adm1_name", "total_flood_obs", "n_flood_days",
     "n_flood_years", "obs_recurring", "obs_unusual"]
].head(15)